# 00 — Check the Final_PROJECT setup on Colab (no GPU needed, ~1 min)

This notebook confirms that the folder and the
DAGHAR data are found, and that the key thesis result files are in place — then re-prints the two
headline numbers straight from those files.

**Paths:** everything is defined in `Final_PROJECT/project_paths.py`. The folder location is detected
automatically; the only setting you may need to change there is `DEFAULT_DATA_PATH` (where DAGHAR's
`standardized_view` lives).

In [1]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("p21", chdir=False)   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH

Mounted at /content/drive
Final_PROJECT paths
  PROJECT_ROOT : /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT
  DATA_PATH    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  component    : p21 -> /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT/01_Contribution1_Generalization_Model/P21_PearsonGraph
  cwd          : /content


In [2]:
import json
checks = {
    "P21 5-seed report":             os.path.join(PP.P21_DIR, "results", "p21_seed_report.json"),
    "P21 per-seed probabilities":    PP.P21_SWEEP,
    "Frozen ConvNet (P18b) probs":   PP.CONVNET_FROZEN_SWEEP,
    "ConvNet reimplementation run":  os.path.join(PP.CONVNET_A_DIR, "results", "run2_nohann"),
    "P24 5-seed report":             os.path.join(PP.P24_DIR, "results", "p24_seed_report_5.json"),
    "P24 edge-budget (5 seeds)":     os.path.join(PP.P24_DIR, "results", "p24_edge_budget_multiseed.json"),
    "P24 checkpoints":               os.path.join(PP.P24_DIR, "results", "checkpoints"),
    "Thesis report":                 os.path.join(PP.PROJECT_ROOT, "00_Thesis_Report", "Thesis_Report.pdf"),
}
for k, v in checks.items():
    print(f"  {'OK ' if os.path.exists(v) else 'MISSING'}  {k:30s} {os.path.relpath(v, PP.PROJECT_ROOT)}")

r = json.load(open(checks["P21 5-seed report"]))["aggregate"]["pearson_topk9_a0.5"]
print(f"\nContribution 1 — locked model, fused LODO accuracy: {r['mean']*100:.2f} ± {r['std']*100:.2f}% "
      f"({r['n_seeds']} seeds; GNN branch alone {r['gnn_only_mean']*100:.2f}%)")

  OK   P21 5-seed report              01_Contribution1_Generalization_Model/P21_PearsonGraph/results/p21_seed_report.json
  OK   P21 per-seed probabilities     01_Contribution1_Generalization_Model/P21_PearsonGraph/results/sweep
  OK   Frozen ConvNet (P18b) probs    01_Contribution1_Generalization_Model/ConvNet_Branch_Frozen/B_Frozen_Predictions_P18b_SourceOnly/results/sweep
  OK   ConvNet reimplementation run   01_Contribution1_Generalization_Model/ConvNet_Branch_Frozen/A_DeepConvLSTM_Reimplementation/results/run2_nohann
  OK   P24 5-seed report              02_Contribution2_SO_Framework/P24_SO_Compression/results/p24_seed_report_5.json
  OK   P24 edge-budget (5 seeds)      02_Contribution2_SO_Framework/P24_SO_Compression/results/p24_edge_budget_multiseed.json
  OK   P24 checkpoints                02_Contribution2_SO_Framework/P24_SO_Compression/results/checkpoints
  OK   Thesis report                  00_Thesis_Report/Thesis_Report.pdf

Contribution 1 — locked model, fused LODO accur

## Which notebook runs what

| Notebook | Component |
|---|---|
| `01_Contribution1_Generalization_Model/P21_PearsonGraph/notebooks/Run_P21_PearsonGraph_Colab.ipynb` | Locked model: graph-config screening, 5-seed training, fusion report |
| `01_…/ConvNet_Branch_Frozen/B_…/notebooks/Run_Frozen_ConvNet_Colab.ipynb` | Frozen ConvNet predictions (inspect / optional re-train) |
| `01_…/ConvNet_Branch_Frozen/A_…/notebooks/Run_ConvNet_DAGHAR_Colab.ipynb` | Standalone DeepConvLSTM reimplementation vs DAGHAR's published numbers |
| `02_Contribution2_SO_Framework/P24_SO_Compression/notebooks/Run_P24_SO_Compression_Colab.ipynb` | SO seed-42 pass: capacity, post-hoc compression, edges, NC/AW |
| `…/Run_P24_MultiSeed_Hardening_Colab.ipynb` | Canonical 5-seed SO numbers + RED/GS edge study |
| `…/Run_P24_CombinedAxis_Colab.ipynb` | Combined axes + Pareto figure |
| `…/Run_P24_SO_Metrics_Recompute_Colab.ipynb` | Macro-F1 / precision / recall for edge & post-hoc variants |